<div style="background-color:#EAEAEA;padding:20px;border-left:5px solid #6C757D;border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;">
<td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1>
<hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de Valencia</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px;font-weight:700;line-height:1.3;margin-top:14px;margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan - <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente - <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none;width:230px;padding-left:25px;text-align:right;vertical-align:top;"><img src="../../logo_uv.png" alt="Universitat de Valencia" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# IceCube -- DeepCore 9-Year Golden Event Oscillation Analysis Data Release
---
Downloads the official IceCube Collaboration public data release accompanying *Measurement of atmospheric neutrino mixing with improved IceCube DeepCore calibration and data processing*, Phys. Rev. D 108, 012014 (2023) (arXiv:2304.12236), from Harvard Dataverse (DOI [10.7910/DVN/B4RITM](https://doi.org/10.7910/DVN/B4RITM)), and caches it verbatim under `data/detector/icecube/raw/` for `tpeanuts.detector.icecube` to consume.

This is the real, event-by-event Monte Carlo release -- not the 2024 CNN-paper release (DOI 10.7910/DVN/U20MMB), which only contains a pre-computed $-2\Delta\ln L$ map, not raw counts/MC usable for an independent forward-model fit.

**Files fetched** (see the release's own `readme.md`, also cached here):
- `data.tab`: real observed event counts per analysis bin (`count`, `pid`, `reco_coszen`, `reco_energy`).
- `mc_nu_nc.tab`, `mc_nue_cc.tab`, `mc_numu_cc.tab`, `mc_nutau_cc.tab`: event-by-event real Monte Carlo (true + reconstructed kinematics, `pdg`, `weight` in GeV cm$^2$ sr, interaction type).
- `mc_mu.tab`: pre-binned real atmospheric-muon background (count + absolute uncertainty).
- `hs_numu_cc.tab`, `hs_nu_nc_nue_cc.tab`, `hs_nutau_cc.tab`: real per-bin, per-$\Delta m^2_{31}$-slice detector-systematics hypersurface corrections.
- `readme.md`: the release's own documentation (column definitions, binning, units).

## Table of Contents

| # | Section |
|---|---|
| [0](#0.-Theoretical-Framework) | **Theoretical Framework** |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Fetch) | **Fetch** |
| [4](#4.-Validate) | **Validate** |
| [5](#5.-Summary) | **Summary** |


## 0. Theoretical Framework

External scientific products are treated as measured or published estimators with explicit provenance, units, binning, and transformation rules. Reproducibility requires that acquisition, parsing, interpolation, normalization, and uncertainty handling preserve the semantics of the primary source.

**References**

The primary dataset publication and repository metadata cited below define the authoritative conventions used in this analysis.


## 1. Libraries

This stage evaluates libraries with explicit provenance, dimensional conventions, numerical controls, and reproducibility checks.


In [1]:
from __future__ import annotations

import time
import urllib.request

import pandas as pd

from tpeanuts.notebooks.notebookConfig import load_notebook_config

config = load_notebook_config()
DATA_DIR = config.data_dir / "detector" / "icecube" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Harvard Dataverse's own file-access API returns 403 for the default
# urllib User-Agent; a browser-like one works.
_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) tpeanuts-thesis-fetch/1.0"}

# File IDs from the Dataverse dataset API
# (https://dataverse.harvard.edu/api/datasets/:persistentId/?persistentId=doi:10.7910/DVN/B4RITM).
FILES = {
    "data.tab": "11646859",
    "mc_nu_nc.tab": "11646850",
    "mc_nue_cc.tab": "11646852",
    "mc_numu_cc.tab": "11646856",
    "mc_nutau_cc.tab": "11646855",
    "mc_mu.tab": "11646851",
    "hs_numu_cc.tab": "11646858",
    "hs_nu_nc_nue_cc.tab": "11646854",
    "hs_nutau_cc.tab": "11646853",
    "readme.md": "11674676",
}

print(f"Torch/pandas ready. Output dir: {DATA_DIR}")

Torch/pandas ready. Output dir: G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\data\detector\icecube\raw


## 2. Paths and Configuration

This stage evaluates paths and configuration with explicit provenance, dimensional conventions, numerical controls, and reproducibility checks.


### 2.1 Paths

Repository-relative paths identify primary-source files, metadata, generated products, and output locations.


### 2.2 Configuration

Acquisition controls, physical units, grids, interpolation rules, validation tolerances, and storage formats are fixed here before data processing.


### 2.3 Local Helpers

Notebook-local functions implement repeated acquisition, parsing, transformation, storage, and validation operations after paths and conventions are fixed.


In [ ]:
def fetch(file_id: str, *, retries: int = 5, timeout: float = 90.0) -> bytes:
    """GET one Dataverse file by numeric id, retrying transient failures with backoff."""
    last_error = None
    for attempt in range(retries):
        try:
            request = urllib.request.Request(f"{BASE_URL}/{file_id}", headers=_HEADERS)
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return response.read()
        except Exception as exc:  # noqa: BLE001 -- retry any transient network error
            last_error = exc
            time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(f"Failed to fetch file id {file_id} after {retries} attempts") from last_error


## 3. Fetch

Skips any file already cached (re-running this notebook is safe/idempotent); set `FORCE_REFRESH = True` to re-download everything.

**Expected results:**<br>
- The fetch products are finite, dimensionally consistent, and reproducible from the cited primary source.
- Parsing, interpolation, normalization, and serialization preserve the published binning and physical measure.
- Validation diagnostics remain within the stated numerical and provenance tolerances.


In [2]:
FORCE_REFRESH = False
BASE_URL = "https://dataverse.harvard.edu/api/access/datafile"




for name, file_id in FILES.items():
    target = DATA_DIR / name
    if target.exists() and not FORCE_REFRESH:
        print(f"skip  (cached) {name}")
        continue
    payload = fetch(file_id)
    target.write_bytes(payload)
    print(f"fetch {len(payload):>10,d} bytes -> {name}")

print("\nDone.")


skip  (cached) data.tab
skip  (cached) mc_nu_nc.tab
skip  (cached) mc_nue_cc.tab
skip  (cached) mc_numu_cc.tab
skip  (cached) mc_nutau_cc.tab
skip  (cached) mc_mu.tab
skip  (cached) hs_numu_cc.tab
skip  (cached) hs_nu_nc_nue_cc.tab
skip  (cached) hs_nutau_cc.tab
skip  (cached) readme.md

Done.


## 4. Validate

Sanity-checks the cached files -- column names, row counts, and a couple of physical ranges -- against the release's own `readme.md` description, before `tpeanuts.detector.icecube.io` relies on them.

**Expected results:**<br>
- The validate products are finite, dimensionally consistent, and reproducible from the cited primary source.
- Parsing, interpolation, normalization, and serialization preserve the published binning and physical measure.
- Validation diagnostics remain within the stated numerical and provenance tolerances.


In [3]:
data = pd.read_csv(DATA_DIR / "data.tab", sep="\t")
assert list(data.columns) == ["count", "pid", "reco_coszen", "reco_energy"]
assert data["count"].sum() > 0
print(f"data.tab             : {len(data):5d} bins, total observed count = {int(data['count'].sum())}")

mc_expected_columns = {
    "reco_energy", "reco_coszen", "pid", "pdg", "true_energy", "true_coszen",
    "weight", "type", "interaction",
    "MaCCQE_linear", "MaCCQE_quad", "MaCCRES_linear", "MaCCRES_quad",
    "Q2", "W", "x", "y",
}
for name in ("mc_nu_nc", "mc_nue_cc", "mc_numu_cc", "mc_nutau_cc"):
    df = pd.read_csv(DATA_DIR / f"{name}.tab", sep="\t")
    assert mc_expected_columns.issubset(df.columns), f"{name}: missing columns"
    assert (df["weight"] > 0).all()
    print(f"{name + '.tab':21s}: {len(df):8d} events, weight in [{df['weight'].min():.3e}, {df['weight'].max():.3e}] GeV cm^2 sr")

mc_mu = pd.read_csv(DATA_DIR / "mc_mu.tab", sep="\t")
assert list(mc_mu.columns) == ["count", "abs_uncertainty", "pid", "reco_coszen", "reco_energy"]
print(f"mc_mu.tab            : {len(mc_mu):5d} bins, total MC muon count = {mc_mu['count'].sum():.2f}")

for name in ("hs_numu_cc", "hs_nu_nc_nue_cc", "hs_nutau_cc"):
    df = pd.read_csv(DATA_DIR / f"{name}.tab", sep="\t")
    assert "deltam31" in df.columns and "intercept" in df.columns
    print(f"{name + '.tab':21s}: {len(df):5d} rows, deltam31 slices = {sorted(df['deltam31'].unique())}")

print("\nPASSED: every cached file matches the release's documented schema.")


data.tab             :   200 bins, total observed count = 21914
mc_nu_nc.tab         :    25346 events, weight in [3.091e-08, 4.011e+03] GeV cm^2 sr


mc_nue_cc.tab        :    39942 events, weight in [8.555e-08, 3.916e+01] GeV cm^2 sr


mc_numu_cc.tab       :   285162 events, weight in [1.942e-08, 1.560e+03] GeV cm^2 sr
mc_nutau_cc.tab      :    46393 events, weight in [3.048e-07, 1.458e+03] GeV cm^2 sr
mc_mu.tab            :   200 bins, total MC muon count = 512.17
hs_numu_cc.tab       :  4000 rows, deltam31 slices = [np.float64(0.0015), np.float64(0.001605), np.float64(0.001711), np.float64(0.001816), np.float64(0.001921), np.float64(0.002026), np.float64(0.002132), np.float64(0.002237), np.float64(0.002342), np.float64(0.002447), np.float64(0.002553), np.float64(0.002658), np.float64(0.002763), np.float64(0.002868), np.float64(0.002974), np.float64(0.003079), np.float64(0.003184), np.float64(0.003289), np.float64(0.003395), np.float64(0.0035)]
hs_nu_nc_nue_cc.tab  :  4000 rows, deltam31 slices = [np.float64(0.0015), np.float64(0.001605), np.float64(0.001711), np.float64(0.001816), np.float64(0.001921), np.float64(0.002026), np.float64(0.002132), np.float64(0.002237), np.float64(0.002342), np.float64(0.002447), np.f

## 5. Summary

The external products, provenance metadata, transformation steps, and validation diagnostics defined above form a reproducible dataset interface for subsequent physical analyses.

**Expected results:**<br>
- Generated files retain the source units, binning, and normalization conventions.
- Integrity and validation checks complete within the stated tolerances.
- The resulting metadata are sufficient to reproduce every transformation from the primary source.
